# Homework Lab 2: Text Preprocessing with Vietnamese
**Overview:** In this exercise, we will build a text preprocessing program for Vietnamese.

Import the necessary libraries. Note that we are using the underthesea library for Vietnamese tokenization. To install it, follow the instructions below. ([link](https://github.com/undertheseanlp/underthesea))

In [1]:
import os,glob
import codecs
import sys
import re
!pip install underthesea
from underthesea import word_tokenize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.7 MB/s eta 0:00:00


## Question 1: Create a Corpus and Survey the Data

The data in this section is partially extracted from the [VNTC](https://github.com/duyvuleo/VNTC) dataset. VNTC is a Vietnamese news dataset covering various topics. In this section, we will only process the science topic from VNTC. We will create a corpus from both the train and test directories. Complete the following program:

- Write `sentences_list` to a file named `dataset_name.txt`, with each element as a document on a separate line.
- Check how many documents are in the corpus.


In [2]:
!pip install rarfile
from rarfile import RarFile
import gdown

def download_dataset(file_name, dataset_name):
  pre_link = "https://github.com/duyvuleo/VNTC/raw/master/Data/10Topics/Ver1.1/"
  rar_file = file_name + ".rar"
  full_link = pre_link + rar_file
  des_link = f"./{dataset_name}/{file_name}/Khoa hoc/" # We have whitespace, so we need to put it in double quote to preserve that

  gdown.download(full_link, rar_file)
  !unrar e {rar_file} "{file_name}/Khoa hoc" "{des_link}" # Use -x for reserving full path of the original unrar files
  os.remove(rar_file)

In [3]:
dataset_name = "VNTC_khoahoc"
download_dataset("Train_Full", dataset_name)
download_dataset("Test_Full", dataset_name)

Downloading...
From: https://github.com/duyvuleo/VNTC/raw/master/Data/10Topics/Ver1.1/Train_Full.rar
To: /content/Train_Full.rar
100%|██████████| 49.2M/49.2M [00:00<00:00, 198MB/s]



UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from Train_Full.rar

Creating    ./VNTC_khoahoc                                            OK
Creating    ./VNTC_khoahoc/Train_Full                                 OK
Creating    ./VNTC_khoahoc/Train_Full/Khoa hoc                        OK
Extracting  ./VNTC_khoahoc/Train_Full/Khoa hoc/KH_VNE_ (29).txt           83%  OK 
Extracting  ./VNTC_khoahoc/Train_Full/Khoa hoc/KH_VNE_ (30).txt           83%  OK 
Extracting  ./VNTC_khoahoc/Train_Full/Khoa hoc/KH_TT_ (1504).txt          83%  OK 
Extracting  ./VNTC_khoahoc/Train_Full/Khoa hoc/KH_VNE_ (33).txt           83%  OK 
Extracting  ./VNTC_khoahoc/Train_Full/Khoa hoc/KH_VNE_ (35).txt           83%  OK 
Extracting  ./VNTC_khoahoc/Train_Full/Khoa hoc/KH_VNE_ (36).txt           83%  OK 
Extracting  ./VNTC_khoahoc/Train_Full/Khoa hoc/KH_VNE_ (38).txt           83%  OK 
Extracting  ./VNTC_khoahoc/Tr

Downloading...
From: https://github.com/duyvuleo/VNTC/raw/master/Data/10Topics/Ver1.1/Test_Full.rar
To: /content/Test_Full.rar
100%|██████████| 77.3M/77.3M [00:00<00:00, 114MB/s]



UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from Test_Full.rar

Creating    ./VNTC_khoahoc/Test_Full                                  OK
Creating    ./VNTC_khoahoc/Test_Full/Khoa hoc                         OK
Extracting  ./VNTC_khoahoc/Test_Full/Khoa hoc/KH_VNE_T_ (2144).txt        68%  OK 
Extracting  ./VNTC_khoahoc/Test_Full/Khoa hoc/KH_VNE_T_ (3).txt           68%  OK 
Extracting  ./VNTC_khoahoc/Test_Full/Khoa hoc/KH_VNE_T_ (4).txt           68%  OK 
Extracting  ./VNTC_khoahoc/Test_Full/Khoa hoc/KH_VNE_T_ (5).txt           68%  OK 
Extracting  ./VNTC_khoahoc/Test_Full/Khoa hoc/KH_VNE_T_ (6).txt           68%  OK 
Extracting  ./VNTC_khoahoc/Test_Full/Khoa hoc/KH_VNE_T_ (7).txt           68%  OK 
Extracting  ./VNTC_khoahoc/Test_Full/Khoa hoc/KH_VNE_T_ (8).txt           68%  OK 
Extracting  ./VNTC_khoahoc/Test_Full/Khoa hoc/KH_VNE_T_ (9).txt           68%  OK 
Extracting 

In [4]:
path = [f'./{dataset_name}/Train_Full/', f'./{dataset_name}/Test_Full/']

# Define folder_list to contain the directories of both train and test sets
folder_list = [os.listdir(path[0]), os.listdir(path[1])]

if os.listdir(path[0]) == os.listdir(path[1]):
    print("train labels = test labels")
else:
    print("train labels differ from test labels")

doc_num = 0
sentences_list = []
meta_data_list = []
for i in range(2):
    for folder_name in folder_list[i]:
        folder_path = path[i] + folder_name
        if folder_name[0] != ".":
            for file_name in glob.glob(os.path.join(folder_path, '*.txt')):
                # Read the file content into f
                f = codecs.open(file_name, 'br')
                # Convert the data to UTF-16 format for Vietnamese text
                file_content = (f.read().decode("utf-16")).replace("\r\n", " ")
                sentences_list.append(file_content.strip())
                f.close
                # Count the number of documents
                doc_num += 1

#### YOUR CODE HERE ####
with open(f"{dataset_name}.txt", "w", encoding = 'utf-8') as f:
  for sentence in sentences_list:
    f.write(sentence + '\n')

print(f"Total documents in the corpus: {doc_num}")
#### END YOUR CODE #####

train labels = test labels
Total documents in the corpus: 3916


## Question 2: Write Preprocessing Functions







### Question 2.1: Write a Function to Clean Text
Hint:
- The text should only retain the following characters: aAàÀảẢãÃáÁạẠăĂằẰẳẲẵẴắẮặẶâÂầẦẩẨẫẪấẤậẬbBcCdDđĐeEèÈẻẺẽẼéÉẹẸêÊềỀểỂễỄếẾệỆfFgGhHiIìÌỉỈĩĨíÍịỊjJkKlLmMnNoOòÒỏỎõÕóÓọỌôÔồỒổỔỗỖốỐộỘơƠờỜởỞỡỠớỚợỢpPqQrRsStTuUùÙủỦũŨúÚụỤưƯừỪửỬữỮứỨựỰvVwWxXyYỳỲỷỶỹỸýÝỵỴzZ0-9(),!?\'\`
- Then trim the whitespace in the input text.

In [5]:
'''
When we write characters in a string, we only need to use backslash "\" before that character, if
1. The character is the same as the one used for quote, such as "\"", or '\''
2. Special characters: "\n", "\r", "\\", ...
'''

def clean_str(string):
    #### YOUR CODE HERE ####
    string = re.sub(r"[^aAàÀảẢãÃáÁạẠăĂằẰẳẲẵẴắẮặẶâÂầẦẩẨẫẪấẤậẬbBcCdDđĐeEèÈẻẺẽẼéÉẹẸêÊềỀểỂễỄếẾệỆfFgGhHiIìÌỉỈĩĨíÍịỊjJkKlLmMnNoOòÒỏỎõÕóÓọỌôÔồỒổỔỗỖốỐộỘơƠờỜởỞỡỠớỚợỢpPqQrRsStTuUùÙủỦũŨúÚụỤưƯừỪửỬữỮứỨựỰvVwWxXyYỳỲỷỶỹỸýÝỵỴzZ0-9(),!?'`]", " ", string)
    string = string.strip()
    string = re.sub("  ", " ", string)
    return string
    #### END YOUR CODE #####

In [6]:
test_text = "Lần Lượt:  Một?  2 Ba (bỐn năM ) 6!  ' `"
print(clean_str(test_text))

Lần Lượt  Một? 2 Ba (bỐn năM ) 6! ' `


### Question 2.2: Write a Function to Convert Text to Lowercase

In [7]:
# make all text lowercase
def text_lowercase(string):
    #### YOUR CODE HERE ####
    string = string.lower()
    return string
    #### END YOUR CODE #####

In [8]:
print(text_lowercase(test_text))

lần lượt:  một?  2 ba (bốn năm ) 6!  ' `


### Question 2.3: Tokenize Words
Hint: Use the `word_tokenize()` function imported above with two parameters: `strings` and `format="text"`.


In [9]:
def tokenize(string):
    #### YOUR CODE HERE ####
    string = word_tokenize(string, format = "text")
    return string
    #### END YOUR CODE #####

In [10]:
print(tokenize(test_text))

Lần_Lượt : Một ? 2 Ba ( bỐn năM ) 6 ! ' `


### Question 2.4: Remove Stop Words
To remove stop words, we use a list of Vietnamese stop words stored in the file `./vietnamese-stopwords.txt`. Complete the following program:
- Check each word in the text (`strings`). If a word is not in the stop words list, add it to `doc_words`.


In [11]:
stopword_url = "https://github.com/stopwords/vietnamese-stopwords/blob/master/vietnamese-stopwords-dash.txt"
stopword_path = "./vietnamese-stopwords.txt"
gdown.download(stopword_url, stopword_path)

with open(stopword_path, "r") as f:
  stopword_list = f.read()

def remove_stopwords(string):
    #### YOUR CODE HERE ####
    tokens = re.split(r"[ ]" , string)
    doc_words = [token for token in tokens if token not in stopword_list]
    return " ".join(doc_words)
    #### END YOUR CODE #####

Downloading...
From: https://github.com/stopwords/vietnamese-stopwords/blob/master/vietnamese-stopwords-dash.txt
To: /content/vietnamese-stopwords.txt
100%|██████████| 1.47k/1.47k [00:00<00:00, 4.08MB/s]


In [12]:
print(remove_stopwords(test_text))

Lần Lượt: Một? Ba (bỐn năM 6! ' `


## Question 2.5: Build a Preprocessing Function
Hint: Call the functions `clean_str`, `text_lowercase`, `tokenize`, and `remove_stopwords` in order, then return the result from the function.


In [13]:
def text_preprocessing(strings):
    #### YOUR CODE HERE ####
    strings = clean_str(strings)
    strings = text_lowercase(strings)
    strings = tokenize(strings)
    doc_words = remove_stopwords(strings)
    return doc_words
    #### END YOUR CODE #####


In [14]:
print(text_preprocessing(test_text))

lần_lượt một ? bốn năm ' `


## Question 3: Perform Preprocessing
Now, we will read the corpus from the file created in Question 1. After that, we will call the preprocessing function for each document in the corpus.

Hint: Call the `text_preprocessing()` function with `doc_content` as the input parameter and save the result in the variable `temp1`.


In [15]:
#### YOUR CODE HERE ####
clean_docs = []

with open(f"{dataset_name}.txt", "r") as f:
  whole_content = f.read()

raw_docs = whole_content.splitlines()

for doc_content in raw_docs:
  temp = text_preprocessing(doc_content)
  clean_docs.append(temp)

#### END YOUR CODE #####
print("\nlength of clean_docs = ", len(clean_docs))
print('clean_docs[0]:\n' + clean_docs[0])


length of clean_docs =  3916
clean_docs[0]:
tìm thấy hóa_thạch động_vật đất_liền cổ nhất một hóa_thạch tìm thấy gần stonehaven đông_scotland đã được xác_định là dấu_tích còn lại của sinh_vật cổ nhất từng sống trên mặt_đất các nhà_khoa_học phỏng_đoán đó là một động_vật nhiều chân dài cm vùi sâu trong một tấm đá bùn sống cách đây 428 triệu năm các chuyên_gia tại bảo_tàng quốc_gia scotland và đại_học yale mỹ đã nghiên_cứu mẫu_vật này trong nhiều tháng họ cho biết nó là bằng_chứng sớm nhất về một sinh_vật sống trên đất_liền khô_ráo chứ không phải dưới biển mike newman một người tìm_kiếm hóa thạch_nghiệp_dư đã tìm thấy hóa_thạch trên bãi bồi của cảng cowie và để ghi_nhận công_lao của ông trong phát_hiện ý_nghĩa này giới khoa_học đã đặt tên loài sinh_vật mới là pneumodesmus_newmani theo tên ông hóa_thạch được xem là lâu_đời hơn triệu năm với động_vật trên cạn cổ nhất từng biết đến trước_kia một sinh_vật giống nhện kỳ_lạ ở aberdeenshire động_vật nhiều chân có các lỗ thở ở hai bên thân là sin

## Question 4: Save Preprocessed Data
Hint: Save the preprocessed data to a file named `dataset_name + '.clean.txt'`, where each document is written on a separate line.


In [16]:
#### YOUR CODE HERE ####
with open(f"{dataset_name}.clean.txt", "w") as f:
  for clean_doc in clean_docs:
    f.write(clean_doc + "\n")
#### YOUR CODE HERE ####